In [1]:
!pip install python-chess torch tqdm -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 89.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
from google.colab import files
uploaded = files.upload()
# Upload games_1200.jsonl, games_1400.jsonl, games_1600.jsonl

Saving games_1200.jsonl to games_1200 (1).jsonl
Saving games_1400.jsonl to games_1400 (1).jsonl
Saving games_1600.jsonl to games_1600 (1).jsonl


In [3]:
import os
os.makedirs("data/processed/lichess", exist_ok=True)
os.makedirs("data/models/behavioral", exist_ok=True)
os.makedirs("data/processed/cache", exist_ok=True)

import shutil
for fname in ["games_1200.jsonl", "games_1400.jsonl", "games_1600.jsonl"]:
    if os.path.exists(fname):
        shutil.move(fname, f"data/processed/lichess/{fname}")
        print(f"Moved {fname}")

Moved games_1200.jsonl
Moved games_1400.jsonl
Moved games_1600.jsonl


In [4]:
import chess
import numpy as np
import torch

PIECE_PLANES = [
    (chess.PAWN,   chess.WHITE), (chess.KNIGHT, chess.WHITE),
    (chess.BISHOP, chess.WHITE), (chess.ROOK,   chess.WHITE),
    (chess.QUEEN,  chess.WHITE), (chess.KING,   chess.WHITE),
    (chess.PAWN,   chess.BLACK), (chess.KNIGHT, chess.BLACK),
    (chess.BISHOP, chess.BLACK), (chess.ROOK,   chess.BLACK),
    (chess.QUEEN,  chess.BLACK), (chess.KING,   chess.BLACK),
]
INPUT_CHANNELS = 13
NUM_MOVES      = 64 * 64

def board_to_tensor(board):
    tensor = np.zeros((INPUT_CHANNELS, 8, 8), dtype=np.float32)
    for i, (pt, color) in enumerate(PIECE_PLANES):
        for sq in board.pieces(pt, color):
            tensor[i, sq // 8, sq % 8] = 1.0
    if board.turn == chess.WHITE:
        tensor[12, :, :] = 1.0
    return torch.tensor(tensor, dtype=torch.float32)

def move_to_index(move):
    return move.from_square * 64 + move.to_square

def index_to_move(index):
    return chess.Move(index // 64, index % 64)

def moves_to_game_samples(moves_uci):
    board, samples = chess.Board(), []
    for uci in moves_uci:
        try:
            move = chess.Move.from_uci(uci)
        except:
            break
        if move not in board.legal_moves:
            break
        samples.append((board_to_tensor(board), move_to_index(move)))
        board.push(move)
    return samples

print("Encoder ready.")

Encoder ready.


In [5]:
import json
from torch.utils.data import Dataset
from tqdm import tqdm
from pathlib import Path

class ChessDataset(Dataset):
    def __init__(self, bracket, max_games=None):
        cache = Path(f"data/processed/cache/dataset_{bracket}.pt")
        if cache.exists():
            print(f"Loading cache: {cache}")
            data = torch.load(cache, weights_only=False)
            self.tensors, self.labels = data["tensors"], data["labels"]
        else:
            path = Path(f"data/processed/lichess/games_{bracket}.jsonl")
            lines = path.read_text().splitlines()
            if max_games:
                lines = lines[:max_games]
            tensors, labels = [], []
            for line in tqdm(lines, desc=f"Encoding [{bracket}]"):
                for t, m in moves_to_game_samples(json.loads(line)["moves"]):
                    tensors.append(t)
                    labels.append(m)
            self.tensors = torch.stack(tensors)
            self.labels  = torch.tensor(labels, dtype=torch.long)
            torch.save({"tensors": self.tensors, "labels": self.labels}, cache)
            print(f"Cached {len(self.tensors):,} positions")
        print(f"[{bracket}] {len(self.tensors):,} positions loaded.")

    def __len__(self): return len(self.tensors)
    def __getitem__(self, i): return self.tensors[i], self.labels[i]

print("Dataset class ready.")

Dataset class ready.


In [6]:
import torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.BatchNorm2d(ch), nn.ReLU(True),
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.BatchNorm2d(ch),
        )
        self.relu = nn.ReLU(True)
    def forward(self, x):
        return self.relu(self.block(x) + x)

class ChessResNet(nn.Module):
    def __init__(self, channels=128, num_blocks=10, dropout=0.3):
        super().__init__()
        self.stem   = nn.Sequential(
            nn.Conv2d(INPUT_CHANNELS, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels), nn.ReLU(True)
        )
        self.tower  = nn.Sequential(*[ResidualBlock(channels) for _ in range(num_blocks)])
        self.policy = nn.Sequential(
            nn.Conv2d(channels, 32, 1, bias=False), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Flatten(),
            nn.Linear(32*8*8, 1024), nn.ReLU(True), nn.Dropout(dropout),
            nn.Linear(1024, NUM_MOVES)
        )
    def forward(self, x):
        return self.policy(self.tower(self.stem(x)))
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

print(f"Model ready. Params: {ChessResNet().count_parameters():,}")

Model ready. Params: 9,270,208


In [7]:
from torch.utils.data import DataLoader, random_split

BRACKETS   = ["1200", "1400", "1600"]
EPOCHS     = 20
BATCH_SIZE = 512        # larger batch = faster on GPU
LR         = 1e-3
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODELS_DIR = Path("data/models/behavioral")

print(f"Device: {DEVICE}")
results = {}

for bracket in BRACKETS:
    print(f"\n{'='*50}\nTraining Elo {bracket}\n{'='*50}")
    ds       = ChessDataset(bracket)
    val_size = int(len(ds) * 0.1)
    trn_ds, val_ds = random_split(ds, [len(ds) - val_size, val_size])
    trn_loader = DataLoader(trn_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model     = ChessResNet().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss()

    best_val_acc, best_path = 0.0, MODELS_DIR / f"chess_bot_{bracket}.pt"

    for epoch in range(1, EPOCHS+1):
        model.train()
        trn_correct = trn_total = 0
        for x, y in trn_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            logits = model(x)
            trn_correct += (logits.argmax(1) == y).sum().item()
            trn_total   += len(y)
        scheduler.step()

        model.eval()
        val_correct = val_total = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                logits = model(x)
                val_correct += (logits.argmax(1) == y).sum().item()
                val_total   += len(y)

        trn_acc = trn_correct / trn_total * 100
        val_acc = val_correct / val_total * 100
        print(f"  Epoch {epoch:02d}/{EPOCHS}  trn_acc={trn_acc:.1f}%  val_acc={val_acc:.1f}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({"bracket": bracket, "epoch": epoch,
                        "val_acc": val_acc, "state_dict": model.state_dict()}, best_path)
            print(f"  Saved best → {best_path}")

    results[bracket] = best_val_acc

print("\nAll brackets done:")
for b, acc in results.items():
    print(f"  Elo {b}: {acc:.1f}%")

Device: cuda

Training Elo 1200


Encoding [1200]: 100%|██████████| 3000/3000 [00:12<00:00, 235.04it/s]


Cached 191,601 positions
[1200] 191,601 positions loaded.
  Epoch 01/20  trn_acc=5.6%  val_acc=9.9%
  Saved best → data/models/behavioral/chess_bot_1200.pt
  Epoch 02/20  trn_acc=13.9%  val_acc=15.5%
  Saved best → data/models/behavioral/chess_bot_1200.pt
  Epoch 03/20  trn_acc=19.0%  val_acc=19.0%
  Saved best → data/models/behavioral/chess_bot_1200.pt
  Epoch 04/20  trn_acc=23.6%  val_acc=21.0%
  Saved best → data/models/behavioral/chess_bot_1200.pt
  Epoch 05/20  trn_acc=27.8%  val_acc=22.3%
  Saved best → data/models/behavioral/chess_bot_1200.pt
  Epoch 06/20  trn_acc=32.4%  val_acc=23.1%
  Saved best → data/models/behavioral/chess_bot_1200.pt
  Epoch 07/20  trn_acc=36.8%  val_acc=23.6%
  Saved best → data/models/behavioral/chess_bot_1200.pt
  Epoch 08/20  trn_acc=41.1%  val_acc=24.5%
  Saved best → data/models/behavioral/chess_bot_1200.pt
  Epoch 09/20  trn_acc=45.6%  val_acc=24.5%
  Epoch 10/20  trn_acc=50.0%  val_acc=24.8%
  Saved best → data/models/behavioral/chess_bot_1200.pt


Encoding [1400]: 100%|██████████| 3000/3000 [00:12<00:00, 235.01it/s]


Cached 189,420 positions
[1400] 189,420 positions loaded.
  Epoch 01/20  trn_acc=6.8%  val_acc=11.0%
  Saved best → data/models/behavioral/chess_bot_1400.pt
  Epoch 02/20  trn_acc=15.4%  val_acc=17.9%
  Saved best → data/models/behavioral/chess_bot_1400.pt
  Epoch 03/20  trn_acc=20.7%  val_acc=19.8%
  Saved best → data/models/behavioral/chess_bot_1400.pt
  Epoch 04/20  trn_acc=25.1%  val_acc=21.2%
  Saved best → data/models/behavioral/chess_bot_1400.pt
  Epoch 05/20  trn_acc=29.1%  val_acc=23.4%
  Saved best → data/models/behavioral/chess_bot_1400.pt
  Epoch 06/20  trn_acc=33.3%  val_acc=23.8%
  Saved best → data/models/behavioral/chess_bot_1400.pt
  Epoch 07/20  trn_acc=37.8%  val_acc=24.4%
  Saved best → data/models/behavioral/chess_bot_1400.pt
  Epoch 08/20  trn_acc=41.9%  val_acc=25.2%
  Saved best → data/models/behavioral/chess_bot_1400.pt
  Epoch 09/20  trn_acc=46.2%  val_acc=25.3%
  Saved best → data/models/behavioral/chess_bot_1400.pt
  Epoch 10/20  trn_acc=50.2%  val_acc=25.8%

Encoding [1600]: 100%|██████████| 3000/3000 [00:14<00:00, 210.92it/s]


Cached 209,402 positions
[1600] 209,402 positions loaded.
  Epoch 01/20  trn_acc=7.4%  val_acc=12.2%
  Saved best → data/models/behavioral/chess_bot_1600.pt
  Epoch 02/20  trn_acc=17.3%  val_acc=17.9%
  Saved best → data/models/behavioral/chess_bot_1600.pt
  Epoch 03/20  trn_acc=22.7%  val_acc=21.4%
  Saved best → data/models/behavioral/chess_bot_1600.pt
  Epoch 04/20  trn_acc=27.2%  val_acc=22.8%
  Saved best → data/models/behavioral/chess_bot_1600.pt
  Epoch 05/20  trn_acc=31.5%  val_acc=24.3%
  Saved best → data/models/behavioral/chess_bot_1600.pt
  Epoch 06/20  trn_acc=35.9%  val_acc=24.9%
  Saved best → data/models/behavioral/chess_bot_1600.pt
  Epoch 07/20  trn_acc=40.4%  val_acc=25.1%
  Saved best → data/models/behavioral/chess_bot_1600.pt
  Epoch 08/20  trn_acc=44.9%  val_acc=26.1%
  Saved best → data/models/behavioral/chess_bot_1600.pt
  Epoch 09/20  trn_acc=49.5%  val_acc=26.3%
  Saved best → data/models/behavioral/chess_bot_1600.pt
  Epoch 10/20  trn_acc=54.2%  val_acc=26.5%

In [8]:
from google.colab import files
files.download("data/models/behavioral/chess_bot_1200.pt")
files.download("data/models/behavioral/chess_bot_1400.pt")
files.download("data/models/behavioral/chess_bot_1600.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>